IMPORT


In [ ]:
import pandas as pd
import os

MessageError: request to https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b1-3l3a5a954afko?authtype=dfs_ephemeral&version=2&dryrun=true&propagate=true&record=false&authuser=0 failed, reason: connect ETIMEDOUT 142.251.216.14:443

CLEANING PLAN


In [2]:
df = pd.read_csv(
    "/content/drive/MyDrive/SE-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
)

# Drop duplicates
df = df.drop_duplicates()

# Drop duplicate symptom column
df = df.drop(columns=["regurgitation.1"])

# Check class balance after dedup
print(df["diseases"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SE-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv'

DROPPING RARE DISEASES AT THE BOTTOM


In [18]:
min_samples = 10
counts = df["diseases"].value_counts()
df = df[df["diseases"].isin(counts[counts >= min_samples].index)]

HANDLING CLASS BALANCE (some have ~1200, others 1 or 0)


In [19]:
# capping samples at 300
df = (
    df.groupby("diseases")
    .apply(lambda x: x.sample(min(len(x), 300)))
    .reset_index(drop=True)
)

/tmp/ipykernel_2044/3400104245.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 300)))


DROPPING ZERO-SYMPTOM ROWS


In [20]:
symptom_cols = df.columns[1:]
zero_rows = df[symptom_cols].sum(axis=1) == 0
print("Zero symptom rows: ", zero_rows.sum())
df = df[~zero_rows]

Zero symptom rows:  0


SAVING DATA


In [29]:
print(f"Final shape: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")
print(df["diseases"].value_counts())

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cleaned_diseases_symptoms.csv", index=False)

Final shape: (100996, 377)
Unique diseases: 587
diseases
acute otitis media                    300
acute kidney injury                   300
white blood cell disease              300
acute bronchospasm                    300
viral exanthem                        300
                                     ... 
pituitary disorder                     10
dislocation of the wrist               10
poisoning due to anticonvulsants       10
idiopathic infrequent menstruation     10
acariasis                              10
Name: count, Length: 587, dtype: int64


In [31]:
print(os.getcwd())
print(os.path.abspath("../data/processed/cleaned_diseases_symptoms.csv"))

/content
/data/processed/cleaned_diseases_symptoms.csv
